In [ ]:
import torch
from segmentation.models.knn_seg import KNNSegmentation
import numpy as np
%load_ext autoreload
%autoreload 2
import pytorch_lightning as pl

# Set random seeds
seed = 42
pl.seed_everything(seed, workers=True)

import matplotlib.pyplot as plt

# Enhanced thesis-ready configuration
plt.rcParams['font.size'] = 14          # Slightly larger base font
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.titlesize'] = 18     # More prominent titles
plt.rcParams['axes.labelsize'] = 14     # Clearer axis labels
plt.rcParams['xtick.labelsize'] = 12    # Readable tick labels
plt.rcParams['ytick.labelsize'] = 12    # Readable tick labels
plt.rcParams['legend.fontsize'] = 12    # Add legend font size
plt.rcParams['figure.titlesize'] = 20   # Overall figure title
plt.rcParams['lines.linewidth'] = 2     # Thicker lines for clarity
plt.rcParams['axes.linewidth'] = 1.2    # Thicker axes
plt.rcParams['grid.alpha'] = 0.3        # Subtle grid if used
plt.rcParams['savefig.dpi'] = 300       # High resolution for print
plt.rcParams['savefig.bbox'] = 'tight'  # Remove extra whitespace
plt.rcParams['figure.figsize'] = (8, 6) # Good default size

<REPO_ROOT>/segmentation/models/vit/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
<REPO_ROOT>/segmentation/models/vit/layers/attention.py:23: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
<HOME>/miniconda3/envs/thesis/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
<REPO_ROOT>/segmentation/models/vit/layers/block.py:30: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")
[rank: 0] Seed set to 42


# DiNoV2 pretrained Results

## Initialization

In [ ]:
from argparse import Namespace
import albumentations as A
import cv2
import torch
import torch.nn as nn
from segmentation.data.retrieval_module_fixed_split import RetrievalDataModuleFixedSplit
from data.dinov2_adaptations import AverageFirstChannelToThird

transform = A.Compose(
    [
        A.PadIfNeeded(
            min_height=532,
            min_width=532,
            # Avoids reflective padding
            border_mode=cv2.BORDER_CONSTANT,
            value=(0, 0, 0),
            p=1,
        ),
        A.CenterCrop(
            532,
            532,
        ),
        AverageFirstChannelToThird(p=1.0),  # Convert 2-channel to 3-channel
    ]
)

data_module = RetrievalDataModuleFixedSplit(
    data_path="./datasets/pretrain_split/",
    batch_size=64,
    num_workers=8,
    transform=transform,
    eval_transform=transform,
    train_dir="train",
    holdout_dir="test",
    classes=["wire", "ball", "wedge", "epoxy"],
    input_resolution=(512, 512),
)

data_module.setup("test")
train_loader = data_module.train_dataloader()
test_loader = data_module.test_dataloader()

encoder = torch.hub.load(
    "facebookresearch/dinov2",
    "dinov2_vits14",
)

encoder = encoder.to("cuda")

class dinoV2Wrapper(nn.Module):
    def __init__(self, encoder):
        super(dinoV2Wrapper, self).__init__()
        self.encoder = encoder

    def forward(self, x):
        return {"x_norm_clstoken": self.encoder(x)}

encoder = dinoV2Wrapper(encoder)

torch.cuda.memory_allocated() / (1024**2)

knn_evaluator = KNNSegmentation(
    encoder=encoder,
    train_loader=train_loader,
    val_loader=test_loader,
    batch_size=32,
    profile_time=True,
 )

Using provided transform for training.


<HOME>/miniconda3/envs/thesis/lib/python3.10/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 6, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Using cache found in <HOME>/.cache/torch/hub/facebookresearch_dinov2_main


In [ ]:
knn_evaluator.evaluate(
    k=4,
    threshold=[0.5, 0.5, 0.5, 0.2])

<REPO_ROOT>/segmentation/models/vit/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
<REPO_ROOT>/segmentation/models/vit/layers/attention.py:23: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
<HOME>/miniconda3/envs/thesis/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
<REPO_ROOT>/segmentation/models/vit/layers/block.py:30: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")
<REPO_ROOT>/segmentation/models/vit/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
<REPO_ROOT>/segmentation/models/vit/layers/attention.py:23: UserWarning: xFormers is ava

Total images loaded: 500
Time taken to move test data to device: 0.05 seconds
Time taken to compute representation: 0.02 seconds
Top distances shape: torch.Size([64, 4])
Time taken to compute nearest neighbors: 1.25 seconds
Using multi-thresholds: [0.5, 0.5, 0.5, 0.2]
Time taken to combine labels: 0.67 seconds
Classwise IoUs: {'iou_wire': 0.2361251562833786, 'iou_ball': 0.5912670493125916, 'iou_wedge': 0.285982608795166, 'iou_epoxy': 0.2839328646659851}
Time taken to compute classwise IoUs: 6.08 seconds
Time taken to move test data to device: 0.04 seconds
Time taken to compute representation: 0.01 seconds
Top distances shape: torch.Size([61, 4])
Time taken to compute nearest neighbors: 0.98 seconds
Using multi-thresholds: [0.5, 0.5, 0.5, 0.2]
Time taken to combine labels: 0.47 seconds
Classwise IoUs: {'iou_wire': 0.166676327586174, 'iou_ball': 0.3498399257659912, 'iou_wedge': 0.20755477249622345, 'iou_epoxy': 0.20241793990135193}
Time taken to compute classwise IoUs: 5.71 seconds
Mean 

{'iou_wire': 0.2014007419347763,
 'iou_ball': 0.4705534875392914,
 'iou_wedge': 0.24676869064569473,
 'iou_epoxy': 0.24317540228366852}

: 